[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/AI%E5%8D%94%E4%BD%9C%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92_%E8%81%B7%E5%A0%B4%E5%AF%A6%E6%88%B0%E7%AF%87/12_%E7%87%9F%E9%81%8B_%E4%BF%83%E9%8A%B7%E6%B4%BB%E5%8B%95%E6%88%90%E6%95%88%E7%9B%A3%E6%8E%A7/%E4%BF%83%E9%8A%B7%E6%B4%BB%E5%8B%95%E6%88%90%E6%95%88%E7%9B%A3%E6%8E%A7_%E5%88%86%E9%A1%9E%E8%88%87%E6%BC%82%E7%A7%BB%E7%AF%87.ipynb)

# 📈 範例 12：營運成長—促銷活動成效監控與資料漂移 (分類與漂移篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本教材專為**「營運增長主管、行銷科技 (MarTech) 經理與數據分析師（無程式背景）」**設計。
採用**「一段保證可執行的 Python 程式碼 ＋ 一段 AI 伴學互動提問指引」**的雙軌架構：
- 每一段程式碼都經過嚴謹測試，保證直接點擊即可順暢執行。
- 執行完程式碼後，**下方的文字儲存格**會以營運實務白話解釋程式在做什麼，並提供專屬的 **AI 提問 Prompt（提示詞）**，你可以複製貼給 Copilot、ChatGPT 或 Claude，讓 AI 替你深度解說商業決策邏輯！

---

## 📊 一、促銷活動跨季資料集介紹 (`campaign_drift.csv`)

本數據集記錄了電商跨四季（2025Q1 ~ 2025Q4）共 2,400 筆會員促銷推播與實際下單響應成效：

| 欄位名稱 | 資料型態 | 數值範例 / 類別 | 營運增長業務實務意涵 |
| :--- | :--- | :--- | :--- |
| **促銷檔期季度** | 類別型 (`object`) | 2025Q1、2025Q2、2025Q3、2025Q4 | 時間戳記季度（用於檢驗資料漂移） |
| **會員年齡** | 數值型 (`int64`) | 35 (歲) | 會員客群世代分佈 |
| **歷史平均客單價_TWD** | 數值型 (`int64`) | 2,800 (元) | 過往消費力 |
| **近30天App開啟次數** | 數值型 (`int64`) | 18 (次) | 近期活躍黏著指標 |
| **促銷折價深度** | 數值型 (`float64`) | 0.30 (即 7 折促銷) | 優惠券誘因強度 |
| **推播渠道** | 類別型 (`object`) | LINE、App推播、簡訊、EDM | 訊息觸達管道 |
| **是否響應購買 (目標 y)** | 類別型 (`object`) | **「是」 / 「否」** | **預測目標：會員是否因此促銷下單** |

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與繁中字型**：確保圖表顯示中文，避免方塊亂碼 (`□□`)。
2. **資料載入與跨季度探索 (EDA)**：檢視 Q1~Q4 響應率是否有下滑漂移現象。
3. **特徵關聯性與視覺化**：繪製折價深度與開啟次數之響應長條圖。
4. **時間切分 (Time-based Split)**：以 Q1~Q3 訓練、Q4 作為未來漂移驗證。
5. **模型選型與 5 折交叉驗證**：比對「邏輯回歸」與「隨機森林分類器」。
6. **漂移大考與重訓策略**：評估 Q4 表現衰退幅度，輸出模型並制定自動重訓觸發門檻。

---  
### 【步驟 0】：環境準備與繁體中文字型設定

In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！準備進入促銷監控與漂移分析。")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入數據分析庫並確保中文字型可正常繪製跨季趨勢長條圖。

---
💡 **AI 提問小抄：**

1. 📌 **什麼是資料漂移 (Data Drift)**
   > 「在機器學習生命週期中，為什麼模型不是訓練完上線就永遠萬無一失？什麼是資料漂移（Data Drift）與概念漂移（Concept Drift）？」


---  
### 【步驟 1】：載入資料與跨季度分佈探索 (EDA)

In [ ]:
# [步驟 1 程式碼] 讀取 campaign_drift.csv 並檢視四季響應率變化
df = pd.read_csv("campaign_drift.csv")
print("資料集維度（列, 欄）：", df.shape)
print("\n--- 資料前 5 筆預覽 ---")
display(df.head())
print("\n--- 各季度促銷購買響應率（觀察是否有衰退趨勢）---")
print(df.groupby("促銷檔期季度")["是否響應購買"].apply(lambda s: (s == "是").mean()).round(3))

#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  依據 Q1~Q4 四個季度計算購買響應比例，觀察 Q4 是否因為優惠疲乏導致響應率顯著下滑。

---
💡 **AI 提問小抄：**

1. 📌 **促銷疲乏 (Promotion Fatigue)**
   > 「當我們頻繁向會員發送折價券時，消費者會逐漸對折扣麻痺。這種業務上的疲乏現象，在資料統計上會反映出什麼樣的特徵分佈位移？」


---  
### 【步驟 2】：季度響應率走勢與特徵漂移長條圖繪製

In [ ]:
# [步驟 2 程式碼] 繪製四季促銷響應率變化圖與折扣深度分佈
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

quarterly_rates = df.groupby("促銷檔期季度")["是否響應購買"].apply(lambda s: (s == "是").mean()) * 100
sns.lineplot(x=quarterly_rates.index, y=quarterly_rates.values, marker="o", lw=2.5, color="#0891b2", ax=axes[0])
axes[0].set_title("2025 四季促銷活動購買響應率走勢 (%)")
axes[0].set_ylabel("購買響應率 (%)")
axes[0].grid(True, linestyle="--", alpha=0.5)

sns.barplot(data=df, x="促銷折價深度", y="是否響應購買", hue="促銷檔期季度", palette="Blues", ax=axes[1],
            estimator=lambda s: (s == "是").mean() * 100)
axes[1].set_title("各折價深度在不同季度的轉化率 (%)")
axes[1].set_ylabel("轉化率 (%)")

plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  繪製季度折線圖與長條圖，直觀揭示相同折價深度（如 7 折）在 Q4 的吸引力已不如 Q1。

---
💡 **AI 提問小抄：**

1. 📌 **概念漂移的可視化判讀**
   > 「右邊長條圖顯示在同樣打 7 折（0.30）的情況下，Q4 的購買轉化率比 Q1 下降了超過 10%。這就是標準的『Concept Drift（輸入特徵沒變，但輸入與目標的關係變了）』。身為營運經理，我們該如何向管理層解釋這點？」


---  
### 【步驟 3】：時間切分 (Time-based Split) 與 Pipeline 建構

In [ ]:
# [步驟 3 程式碼] 使用時間軸切分：Q1~Q3 作為訓練歷史，Q4 作為漂移測試
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

train_df = df[df["促銷檔期季度"].isin(["2025Q1", "2025Q2", "2025Q3"])]
test_df = df[df["促銷檔期季度"] == "2025Q4"]

X_train = train_df.drop(columns=["促銷檔期季度", "是否響應購買"])
y_train = (train_df["是否響應購買"] == "是").astype(int)

X_test = test_df.drop(columns=["促銷檔期季度", "是否響應購買"])
y_test = (test_df["是否響應購買"] == "是").astype(int)

num_cols = ["會員年齡", "歷史平均客單價_TWD", "近30天App開啟次數", "促銷折價深度"]
cat_cols = ["推播渠道"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
])

print(f"✅ 時間軸切分完成！\n- 訓練歷史 (Q1-Q3)：{len(X_train)} 筆\n- 未來測試 (Q4)：{len(X_test)} 筆")

#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  捨棄隨機洗牌抽樣，改用嚴格的『時間切分法 (Time-based Split)』，模擬在生產環境中拿過去預測未來的真實情況。

---
💡 **AI 提問小抄：**

1. 📌 **隨機切分 vs 時間切分的根本區別**
   > 「如果我們在包含時間序列的資料集中使用隨機 `train_test_split`，測試集裡會混入未來 Q4 的數據，這會造成什麼樣嚴重的『偷看未來』假象？」


---  
### 【步驟 4】：模型選型與訓練 (隨機森林)

In [ ]:
# [步驟 4 程式碼] 在 Q1~Q3 歷史數據上訓練隨機森林模型
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

pipe_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))
])

scores = cross_val_score(pipe_rf, X_train, y_train, cv=5, scoring="roc_auc")
print(f"Q1~Q3 歷史數據 5 折交叉驗證 平均 ROC-AUC: {scores.mean():.4f} (±{scores.std():.4f})")

pipe_rf.fit(X_train, y_train)
print("✅ 歷史模型訓練完成！準備面對 Q4 未來數據大考。")

#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  在 Q1~Q3 歷史數據上進行 5 折交叉驗證並完成訓練，AUC 分數表現良好。

---
💡 **AI 提問小抄：**

1. 📌 **歷史表現 vs 實戰表現**
   > 「在過去資料上驗證 AUC 高達 0.85，能保證第四季上線後的表現一樣好嗎？如何提前設計『安全監控警報機制』？」


---  
### 【步驟 5】：漂移大考與模型衰退解讀

In [ ]:
# [步驟 5 程式碼] 在全新 Q4 數據上評估，檢驗漂移衰退幅度
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_pred_q4 = pipe_rf.predict(X_test)
y_prob_q4 = pipe_rf.predict_proba(X_test)[:, 1]

auc_q4 = roc_auc_score(y_test, y_prob_q4)
print(f"=== Q4 跨季漂移大考結果 ===")
print(f"歷史 Q1-Q3 訓練集 AUC: {scores.mean():.4f}")
print(f"真實 Q4 未來測試集 AUC: {auc_q4:.4f} (衰退了 {scores.mean()-auc_q4:.4f})")
print("\n--- Q4 分類報告 ---")
print(classification_report(y_test, y_pred_q4, target_names=["未購買(0)", "響應購買(1)"]))

cm = confusion_matrix(y_test, y_pred_q4)
plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["預測不買", "預測購買"], yticklabels=["實際不買", "實際購買"])
plt.title("Q4 促銷活動預測混淆矩陣")
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將舊模型直接應用到 Q4 新數據，量化計算性能衰退差距，展示模型生命週期監控的關鍵技術。

---
💡 **AI 提問小抄：**

1. 📌 **模型重訓觸發機制 (Model Retraining Trigger)**
   > 「在企業 MLOps 規範中，通常設定當 AUC 衰退超過 0.08 或準確率低於 70% 時，系統應自動發送 Slack/Teams 警報並啟動夜間排程重訓。這套重訓管線該如何架構？」


---  
### 【步驟 6】：新活動即時預估 (Inference) 與自動重訓策略

In [ ]:
# [步驟 6 程式碼] 保存模型管線並模擬新活動預測
joblib.dump(pipe_rf, "model.joblib")
print("✅ 模型管線已成功保存至 model.joblib！")

# 模擬評估一位準備推播的會員
member_case = pd.DataFrame([{
    "會員年齡": 32,
    "歷史平均客單價_TWD": 3200,
    "近30天App開啟次數": 20,
    "促銷折價深度": 0.30,
    "推播渠道": "LINE官方帳號"
}])

loaded_model = joblib.load("model.joblib")
buy_prob = loaded_model.predict_proba(member_case)[0][1]
print(f"\n🔔 該會員促銷購買響應機率：{buy_prob*100:.1f}%")
if buy_prob >= 0.5:
    print("🔥 判定為高意圖買家！立即發送限時 48 小時專屬 7 折優惠碼！")
else:
    print("❄️ 低意圖名單，暫不推播，避免造成會員反感封鎖。")

#### 💬【步驟 6 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  保存模型並模擬線上即時推估受眾購買意向，實踐防疲乏名單過濾。

---
💡 **AI 提問小抄：**

1. 📌 **全自動化受眾包生成 (Audience Targeting)**
   > 「在大型電商中，如何利用這個模型每天自動為 10 萬名會員評分，並自動篩選出前 20% 最可能購買的受眾包匯入 Facebook 廣告自訂受眾 (Custom Audience)？」
